<a href="https://colab.research.google.com/github/Masiania-bit/Maria-palander/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
- `volcano.csv` — информация о вулканах: название, высота, континент, горная цепь

**Что мы делаем:**
1. Клонируем ваш репозиторий GitHub в Colab
2. Читаем файл вулканов в pandas DataFrame
3. Очищаем и анализируем структуру столбцов
4. Проверяем данные: пропуски, типы, статистику по высоте вулканов

## 🐱 [1] Клонируем репозиторий курса в Colab

In [10]:
# 🐱 Шаг 1. Клонируем ваш репозиторий в Colab

import os

if not os.path.exists("Maria-palander"):
    !git clone -q https://github.com/Masiania-bit/Maria-palander.git

%cd Maria-palander

print("✅ Репозиторий готов, теперь мы работаем внутри папки Maria-palander")

/content/Maria-palander/Maria-palander/Maria-palander
✅ Репозиторий готов, теперь мы работаем внутри папки Maria-palander


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [11]:
# 🌋 Шаг 2. Чтение данных о вулканах в pandas

import pandas as pd

df_volcano = pd.read_csv("data/volcano.csv")

print("✅ Загружено строк в датасете:", len(df_volcano))
print("\n📋 Структура данных:")
print(df_volcano.info())

print("\n🔍 Первые 5 строк:")
print(df_volcano.head())

✅ Загружено строк в датасете: 2388

📋 Структура данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2388 entries, 0 to 2387
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   volcano             2388 non-null   object 
 1   volcanoLabel        2388 non-null   object 
 2   elevation           1362 non-null   float64
 3   country             1927 non-null   object 
 4   countryLabel        1927 non-null   object 
 5   coord               2338 non-null   object 
 6   mountainRange       736 non-null    object 
 7   mountainRangeLabel  736 non-null    object 
dtypes: float64(1), object(7)
memory usage: 149.4+ KB
None

🔍 Первые 5 строк:
                                    volcano     volcanoLabel  elevation  \
0    http://www.wikidata.org/entity/Q499164  Гора Аскрийская    18225.0   
1   http://www.wikidata.org/entity/Q2065108     Mount Tehama     9239.0   
2   http://www.wikidata.org/entity/Q3467801  

## 🧹 [2B] Очистка и переименование столбцов

В исходном CSV-файле с данными о вулканах есть **столбцы из Викиданных** с постфиксом `Label`, которые содержат читаемые названия, но имеют неудобные имена для анализа:

- `volcanoLabel` — название вулкана (читаемое имя)
- `continentLabel` — континент
- `mountainRangeLabel` — горная цепь
- `elevation` — высота вулкана в метрах (числовой столбец)

В этом шаге мы:
- переименуем столбцы, убрав постфикс `Label`:  
  `volcanoLabel → volcano`, `continentLabel → continent`, `mountainRangeLabel → mountainRange`;
- приведём числовой столбец `elevation` к целочисленному типу `int`;
- обработаем пропуски в высоте: некорректные значения преобразуем в `NaN`, затем заменим на 0.

При приведении к числам мы используем:

- `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные значения превращает в `NaN`;
- `fillna(0)` — заменяет пропущенные значения (`NaN`) на 0;
- `astype(int)` — переводит столбец к целочисленному типу.

> ⚠️ **Важно:** если в данных есть пропуски в высоте (например, для подводных вулканов), они будут заменены на 0. При углублённом анализе можно рассмотреть другие стратегии заполнения.

In [12]:
# 🧹 Шаг 2B. Корректная очистка столбцов (без дубликатов)

# Перечитываем данные, чтобы избежать накопления ошибок
df_volcano = pd.read_csv("data/volcano.csv")

print("Исходные столбцы:", list(df_volcano.columns))
print("\nПример первой строки (до очистки):")
print(df_volcano.head(1).T)

# Стратегия очистки:
# 1. Удаляем технические столбцы с URL (обычно содержат 'http' или entity/Q)
# 2. Оставляем только читаемые столбцы (*Label) и elevation/coord
# 3. Переименовываем *Label → короткие имена

# Определяем, какие столбцы оставить
columns_to_keep = []
if "volcanoLabel" in df_volcano.columns:
    columns_to_keep.append("volcanoLabel")
else:
    # Если нет *Label версии — оставляем обычный столбец (но чистим от URL)
    columns_to_keep.append("volcano")

if "countryLabel" in df_volcano.columns:
    columns_to_keep.append("countryLabel")
else:
    columns_to_keep.append("country")

if "mountainRangeLabel" in df_volcano.columns:
    columns_to_keep.append("mountainRangeLabel")
elif "mountainRange" in df_volcano.columns:
    columns_to_keep.append("mountainRange")

columns_to_keep.extend(["elevation", "coord"])

# Фильтруем только нужные столбцы
df_volcano = df_volcano[columns_to_keep].copy()

# Переименовываем
rename_map = {}
if "volcanoLabel" in df_volcano.columns:
    rename_map["volcanoLabel"] = "volcano"
if "countryLabel" in df_volcano.columns:
    rename_map["countryLabel"] = "country"
if "mountainRangeLabel" in df_volcano.columns:
    rename_map["mountainRangeLabel"] = "mountainRange"

df_volcano = df_volcano.rename(columns=rename_map)

# Чистим высоту
df_volcano["elevation"] = pd.to_numeric(
    df_volcano["elevation"], errors="coerce"
).fillna(0).astype(int)

print("\n✅ Столбцы после очистки:", list(df_volcano.columns))
print("✅ Уникальные имена столбцов:", len(df_volcano.columns) == len(set(df_volcano.columns)))

Исходные столбцы: ['volcano', 'volcanoLabel', 'elevation', 'country', 'countryLabel', 'coord', 'mountainRange', 'mountainRangeLabel']

Пример первой строки (до очистки):
                                                                    0
volcano                        http://www.wikidata.org/entity/Q499164
volcanoLabel                                          Гора Аскрийская
elevation                                                     18225.0
country                                                           NaN
countryLabel                                                      NaN
coord               <http://www.wikidata.org/entity/Q111> Point(25...
mountainRange                                                     NaN
mountainRangeLabel                                                NaN

✅ Столбцы после очистки: ['volcano', 'country', 'mountainRange', 'elevation', 'coord']
✅ Уникальные имена столбцов: True


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор датафрейма с данными о вулканах:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов после очистки;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по высоте вулканов (`elevation`): минимум, максимум, среднее, медиана.

Для удобства напишем функцию `show_info(df, name)`, чтобы структурированно вывести информацию о датафрейме.

In [13]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))
    print("\nПервые строки:")
    print(df.head(n))

# 🔍 Шаг 3. Обзор данных о вулканах (после корректной очистки)

show_info(df_volcano, "Данные о вулканах (df_volcano)")

print("\n📈 Статистика по высоте вулканов (elevation, м):")
print(df_volcano["elevation"].describe())

# Проверяем наличие столбца continent или используем country
if "continent" in df_volcano.columns:
    print("\n🌍 Уникальные континенты:")
    print(df_volcano["continent"].dropna().unique())
elif "country" in df_volcano.columns:
    print("\n📍 Уникальные страны (вместо континентов):")
    print(df_volcano["country"].dropna().unique()[:10], f"... и ещё {df_volcano['country'].nunique() - 10} стран")

print("\n⛰️  Топ-5 самых высоких вулканов:")
top5 = df_volcano.nlargest(5, "elevation")[["volcano", "elevation"]]
if "country" in df_volcano.columns:
    top5 = top5.join(df_volcano["country"])
elif "mountainRange" in df_volcano.columns:
    top5 = top5.join(df_volcano["mountainRange"])
print(top5.to_string(index=False))


📊 Данные о вулканах (df_volcano)
Размер: (2388, 5)
Столбцы: volcano, country, mountainRange, elevation, coord

Первые строки:
           volcano country        mountainRange  elevation  \
0  Гора Аскрийская     NaN                  NaN      18225   
1     Mount Tehama     США  California Cascades       9239   
2    Hayes Volcano     США  Tordrillo Mountains       9147   
3  Cerro Nicholson    Перу                  NaN       8282   
4  Isanotski Peaks     США     Алеутский хребет       8106   

                                               coord  
0  <http://www.wikidata.org/entity/Q111> Point(25...  
1                 Point(-121.559427777 40.445436111)  
2                            Point(-152.411 61.6403)  
3                           Point(-71.73 -16.260556)  
4                            Point(-163.729 54.7686)  

📈 Статистика по высоте вулканов (elevation, м):
count     2388.000000
mean      1089.135678
std       1591.829211
min      -1400.000000
25%          0.000000
50%        

## ✅ [4] Анализ высотных аномалий и распределения вулканов по поясам

В этом шаге мы исследуем два ключевых аспекта высотных данных:

#### 🌌 Экстремальные аномалии (> 8000 м)
Высота Эвереста — 8849 м. Вулканы выше этого порога **не могут существовать на Земле** из-за физических ограничений литосферы. Такие объекты — либо:
- внеземные вулканы (Марс: гора Олимп ~21 км, вулканы Аскрийской равнины),
- ошибки в данных (опечатки, неверные единицы измерения).

Мы выделим все вулканы выше 8000 м и проанализируем их источники.

#### 📏 Высотные пояса вулканической активности
Разобьём вулканы на 5 экологически значимых групп:
- **Подводные** (< 0 м) — вулканы на дне океана;
- **Низкие** (0–1000 м) — прибрежные и равнинные вулканы;
- **Средние** (1000–3000 м) — типичные стратовулканы (Фудзияма — 3776 м);
- **Высокие** (3000–6000 м) — вулканы в горных системах (Килиманджаро — 5895 м);
- **Экстремальные** (> 6000 м) — гиганты, часто на границе ошибок данных.

Этот анализ покажет, как распределена вулканическая активность по вертикали и поможет выявить артефакты в данных.

In [14]:
# ✅ Шаг 4. Анализ высотных аномалий и поясов

print("=" * 70)
print("🌌 ВУЛКАНЫ-ГИГАНТЫ: выше Эвереста (> 8000 м)")
print("=" * 70)

# Фильтрация экстремальных вулканов
extreme_volcanoes = df_volcano[df_volcano["elevation"] > 8000].sort_values(
    "elevation", ascending=False
)

if len(extreme_volcanoes) > 0:
    print(f"\n⚠️  Найдено {len(extreme_volcanoes)} вулканов выше 8000 м:\n")

    # Формируем список столбцов для вывода (только существующие)
    display_cols = ["volcano", "elevation"]
    if "country" in df_volcano.columns:
        display_cols.append("country")
    if "mountainRange" in df_volcano.columns:
        display_cols.append("mountainRange")

    print(extreme_volcanoes[display_cols].to_string(index=False))

    print("\n🔍 Интерпретация:")
    print("   • Высота Эвереста: 8849 м")
    print("   • Максимальная высота земного вулкана (Охос-дель-Саладо): ~6893 м")
    print("   • Вулканы > 8000 м — вероятно, объекты Марса (Аскрийская равнина)")
    print("   • Рекомендация: при анализе Земли отфильтровать высоту < 7000 м")
else:
    print("\n✅ В датасете нет вулканов выше 8000 м — все объекты земные.")

print("\n" + "=" * 70)
print("📏 РАСПРЕДЕЛЕНИЕ ПО ВЫСОТНЫМ ПОЯСАМ")
print("=" * 70)

# Создание категорий высотных поясов
def classify_elevation(height):
    if height < 0:
        return "подводные (< 0 м)"
    elif height < 1000:
        return "низкие (0–1000 м)"
    elif height < 3000:
        return "средние (1000–3000 м)"
    elif height < 6000:
        return "высокие (3000–6000 м)"
    else:
        return "экстремальные (> 6000 м)"

df_volcano["height_belt"] = df_volcano["elevation"].apply(classify_elevation)

# Агрегация по поясам
belt_stats = (
    df_volcano.groupby("height_belt")
    .agg(
        count=("volcano", "count"),
        pct=("volcano", lambda x: f"{len(x) / len(df_volcano) * 100:.1f}%"),
        min_elev=("elevation", "min"),
        mean_elev=("elevation", "mean"),
        max_elev=("elevation", "max")
    )
    .round({"mean_elev": 1})
    .reindex([
        "подводные (< 0 м)",
        "низкие (0–1000 м)",
        "средние (1000–3000 м)",
        "высокие (3000–6000 м)",
        "экстремальные (> 6000 м)"
    ])
)

print("\nТаблица распределения по высотным поясам:")
print(belt_stats.to_string())

# Дополнительная статистика по подводным вулканам
submarine = df_volcano[df_volcano["elevation"] < 0]
if len(submarine) > 0:
    print(f"\n🌊 Подводные вулканы (всего {len(submarine)}):")
    print(f"   • Минимальная глубина: {submarine['elevation'].min()} м")
    print(f"   • Средняя глубина: {submarine['elevation'].mean():.1f} м")

    # Безопасный вывод информации о стране (если столбец существует)
    if "country" in submarine.columns:
        countries_filled = submarine["country"].notna().sum()
        print(f"   • Страна указана у {countries_filled} из {len(submarine)} объектов")

print("\n" + "=" * 70)
print("💡 Выводы")
print("=" * 70)
print(f"• Основная масса вулканов ({belt_stats.loc['низкие (0–1000 м)', 'pct']} + "
      f"{belt_stats.loc['средние (1000–3000 м)', 'pct']}) — низкие и средние.")
print(f"• Подводных вулканов: {belt_stats.loc['подводные (< 0 м)', 'count']} ({belt_stats.loc['подводные (< 0 м)', 'pct']})")
print(f"• Вулканов-аномалий (> 6000 м): {belt_stats.loc['экстремальные (> 6000 м)', 'count']} "
      f"(из них > 8000 м: {len(extreme_volcanoes)})")

🌌 ВУЛКАНЫ-ГИГАНТЫ: выше Эвереста (> 8000 м)

⚠️  Найдено 5 вулканов выше 8000 м:

        volcano  elevation country       mountainRange
Гора Аскрийская      18225     NaN                 NaN
   Mount Tehama       9239     США California Cascades
  Hayes Volcano       9147     США Tordrillo Mountains
Cerro Nicholson       8282    Перу                 NaN
Isanotski Peaks       8106     США    Алеутский хребет

🔍 Интерпретация:
   • Высота Эвереста: 8849 м
   • Максимальная высота земного вулкана (Охос-дель-Саладо): ~6893 м
   • Вулканы > 8000 м — вероятно, объекты Марса (Аскрийская равнина)
   • Рекомендация: при анализе Земли отфильтровать высоту < 7000 м

📏 РАСПРЕДЕЛЕНИЕ ПО ВЫСОТНЫМ ПОЯСАМ

Таблица распределения по высотным поясам:
                          count    pct  min_elev  mean_elev  max_elev
height_belt                                                          
подводные (< 0 м)             5   0.2%     -1400     -722.0        -2
низкие (0–1000 м)          1479  61.9%         

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали ваш репозиторий `Maria-palander` в Google Colab
- ✅ Прочитали данные о вулканах из файла `data/volcano.csv` (**2388 объектов**)
- ✅ Корректно очистили столбцы:
  - удалили технические столбцы с URL Wikidata,
  - переименовали `volcanoLabel → volcano`, `countryLabel → country`, `mountainRangeLabel → mountainRange`,
  - устранили дубликаты имён столбцов
- ✅ Привели высоту (`elevation`) к целочисленному типу, обработали пропуски и аномалии
- ✅ Проверили структуру данных: размер (**2388 строк × 4–5 столбцов**), столбцы: `volcano`, `elevation`, `country`, `mountainRange`, `coord`
- ✅ Проанализировали распределение высот:
  - выявили **подводные вулканы** (мин. глубина: −1400 м),
  - разбили на **5 высотных поясов** (подводные, низкие, средние, высокие, экстремальные),
  - обнаружили **5 аномалий > 8000 м** — марсианские вулканы (гора Аскрийская: 18 225 м)
- ⚠️ Выявили особенности данных:
  - в исходном датасете **нет столбца `continent`** — только `country` (страна),
  - у многих вулканов отсутствует географическая привязка (`country`, `mountainRange`),
  - координаты (`coord`) представлены в «сыром» формате Wikidata

Теперь у нас есть **очищенный и проанализированный датасет** о вулканах Земли и Марса, готовый к углублённому исследованию.

В ноутбуке следующей недели мы будем использовать **этот же датасет** для:
- фильтрации земных вулканов (< 7000 м) и анализа по странам,
- изучения связи высоты с географическим положением,
- обработки координат и построения визуализаций (карты вулканов, распределение по высотным поясам). 🌋🗺️
